# Quantra — Train on Colab

One universal **PPO** policy built to **repeatedly pass FTMO-style challenges** (2.5% daily target / 4% trailing wall) — not to maximize PnL.

An episode is **N trading days on ONE continuous account** (C10): the balance carries forward day to day, a day that hits the wall is **locked out for the rest of that day** and resets at midnight (it does **not** end the episode), and the episode ends only when the N days are exhausted or the account is blown. A **large end-of-day penalty** (C11) fires whenever the bot misses that day's target, so it learns **consistency**, not just survival.

> ⚠️ **Pick a CPU runtime** (Runtime → Change runtime type → CPU, or High-RAM). The locked net is a tiny **3×256 MLP (~207 inputs)** — the bottleneck is CPU env-stepping, not the GPU. The optimizer below will *prove* whether a GPU is worth it (it usually isn't), so you don't waste paid GPU hours.

## 1. Clone the repo

In [ ]:
# The latest code lives on this BRANCH (change to "main" once merged). Without -b, Colab would clone
# the default branch and miss the new modules (barbershop_runner, repo_graph, the dashboard screens).
!git clone -b claude/focused-faraday-if1ue7 --single-branch https://github.com/monty313/RL-model-trading-bot-ppo-mlp_Claude-.git quantra_repo
%cd quantra_repo
!git rev-parse --abbrev-ref HEAD

## 2. Install dependencies
Colab already ships torch, numpy, pandas, scikit-learn, matplotlib. We only add the extras.

In [ ]:
!pip install -q gdown pyarrow psutil optuna seaborn statsmodels gymnasium tqdm
# nvidia-ml-py is only needed if you chose a GPU runtime:
import torch
if torch.cuda.is_available():
    !pip install -q nvidia-ml-py

## 3. Mount Google Drive (price data)
Your 4 symbols (MT5 1m, ~5yr) live in the `rl-trading-data` folder. Mounting lets the data loader read them without re-downloading 500 MB each session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Expected path: /content/drive/MyDrive/rl-trading-data/<SYMBOL>_M1_*.csv
!ls -la "/content/drive/MyDrive/rl-trading-data" 2>/dev/null || echo "Adjust the folder path if your Drive layout differs; gdown-by-ID fallback is built into the loader."

## 4. Race the hardware (CPU vs GPU) and size to ~80%
This is the money-saver: it benchmarks both devices on the real four-head workload, picks the faster one (preferring CPU on a near-tie), scales parallel envs to ~80% utilisation, and reports what it measured.

In [ ]:
from quantra.runtime import RuntimeConfig, plan, print_report, UtilizationMonitor, ensure_dirs

ensure_dirs()
cfg = RuntimeConfig()
with UtilizationMonitor(interval=0.25) as mon:
    hw = plan(state_dim=cfg.nominal_state_dim, hw=cfg.hardware)
util = mon.stop()

print_report(hw)
print('Utilisation during race:', util.render())

## 5. HYPERPARAMETERS — all visible + editable BEFORE any run
Every knob the training run uses is surfaced here. **🔴 = LOCKED** (visible but read-only; changing needs Monty's sign-off). Edit the values, then build the env/trainer from the objects this cell constructs — nothing downstream is hardcoded.

In [ ]:
# ── HYPERPARAMETERS (edit here; gamma/lambda are 🔴 LOCKED — visible only) ──
from quantra.runtime import config as qcfg
from quantra.learning_system.trainer.gae import GAMMA, LAMBDA            # 🔴 0.997 / 0.97
from quantra.learning_system.trainer.trainer import TrainConfig
from quantra.learning_system.trainer.scheduler import AggressionRanges

# --- Episode length (C10: N trading days on ONE continuous account) ----------
TRAINING_DAYS = qcfg.TRAINING_DAYS     # default 180 (~6 months) for a full training run
EVAL_DAYS     = qcfg.EVAL_DAYS         # default 8 for a Barbershop/eval window
EPISODE_DAYS  = TRAINING_DAYS          # what THIS run uses (swap to EVAL_DAYS for a short eval)

# --- Challenge numbers (what "passing" means; → make_challenge) --------------
DAILY_TARGET_PCT   = 2.5               # daily profit goal (% of that day's OPENING balance)
DAILY_RISK_PCT     = 4.0               # daily trailing-wall risk
FTMO_MODE          = True              # ON: 2-phase auto-flat at target. OFF: target is the aim
STOP_FOR_DAY       = False             # OFF-mode: bank + lock out the day at target (else run on)
FAILED_DAY_PENALTY = 5.0               # C11 weight — LARGE end-of-day penalty, ∝ shortfall from target
PERMANENT_DD_PCT   = 10.0              # -10% max wall — OBSERVATION only (C12), NOT enforced

# --- Reward weights (C16: plain-English, operator-tunable; math/structure unchanged) ---
# Layer 0 net-PnL is the dominant outcome base E8 protects (keep net_pnl_weight = 1.0). The four
# shaping weights are small "whispers". daily_progress_weight is the consistency driver (matters most).
NET_PNL_WEIGHT         = 1.0           # 🔴 keep 1.0 (E8 Layer-0 dominance)
STEP_PNL_WEIGHT        = 1e-4          # small per-bar in-position bonus
DAILY_PROGRESS_WEIGHT  = 1e-3          # the consistency driver (raised from 1e-4)
DRAWDOWN_PAIN_WEIGHT   = 5e-4          # pain-zone penalty near the wall
DRAWDOWN_PAIN_STEEPNESS= 4.0           # exponential steepness of the pain ramp
TRADE_QUALITY_WEIGHT   = 5e-5          # close-quality / target-progress whisper

# --- Enforcement / safety toggles -------------------------------------------
TRAINING_WHEELS = qcfg.TRAINING_WHEELS  # counter-trend OPEN blocks (True = on)
TRAINING_PHASE  = qcfg.TRAINING_PHASE   # PHASE_FREE (signals observation-only) / PHASE_CONSTRAINED

# --- PPO discount (🔴 LOCKED — shown for visibility, do not change) ----------
GAMMA_LOCKED  = GAMMA                  # 🔴 0.997  (the "math of patience")
LAMBDA_LOCKED = LAMBDA                 # 🔴 0.97

# --- Aggression dial RANGES (the missed-opportunity scheduler picks within these) ---
ENTROPY_RANGE = (0.03, 0.08)
CLIP_RANGE    = (0.25, 0.35)
LR_RANGE      = (5e-4, 1e-3)
EPOCHS_RANGE  = (10, 15)

# --- Rollout / optimization --------------------------------------------------
ROLLOUT_SIZE   = 512
MINIBATCH_SIZE = 64
VALUE_COEF     = 0.5

# Build the REAL config objects the env + Trainer consume (nothing hardcoded downstream).
CHALLENGE  = qcfg.make_challenge(daily_target_pct=DAILY_TARGET_PCT, daily_risk_pct=DAILY_RISK_PCT,
                                 ftmo_mode=FTMO_MODE, stop_for_day=STOP_FOR_DAY,
                                 permanent_dd_pct=PERMANENT_DD_PCT,
                                 failed_day_penalty=FAILED_DAY_PENALTY)
REWARD_CFG = qcfg.RewardConfig(net_pnl_weight=NET_PNL_WEIGHT, step_pnl_weight=STEP_PNL_WEIGHT,
                               daily_progress_weight=DAILY_PROGRESS_WEIGHT,
                               drawdown_pain_weight=DRAWDOWN_PAIN_WEIGHT,
                               drawdown_pain_steepness=DRAWDOWN_PAIN_STEEPNESS,
                               trade_quality_weight=TRADE_QUALITY_WEIGHT,
                               failed_day_penalty=FAILED_DAY_PENALTY)
TRAIN_CFG  = TrainConfig(rollout_size=ROLLOUT_SIZE, minibatch=MINIBATCH_SIZE, value_coef=VALUE_COEF)
AGG_RANGES = AggressionRanges(entropy=ENTROPY_RANGE, clip=CLIP_RANGE, lr=LR_RANGE, epochs=EPOCHS_RANGE)
# (the env is built as TradingEnv(..., challenge=CHALLENGE, reward_cfg=REWARD_CFG, episode_days=EPISODE_DAYS))

# --- C19: OVERRIDES = ONLY the knobs that differ from baseline -> auto-names + reproduces the policy ---
# build_overrides_dict diffs the live config objects above against the dataclass defaults. The result
# NAMES the policy (auto_name) and is saved verbatim in its manifest by the Barbershop loop (Cell 6),
# so the Leaderboard can tell policies apart and every run is reproducible. Edit knobs ABOVE, not here.
from quantra.learning_system.policy_registry import auto_name
OVERRIDES = qcfg.build_overrides_dict(challenge=CHALLENGE, reward=REWARD_CFG, train=TRAIN_CFG,
                                      training_phase=TRAINING_PHASE, training_wheels=TRAINING_WHEELS)
POLICY_NAME, _name_basis = auto_name(OVERRIDES)   # base_policy is supplied by RESUME_FROM in Barbershop

print(f"Episode: {EPISODE_DAYS} days/episode (one continuous account) | failed-day penalty={FAILED_DAY_PENALTY}")
print(f"Challenge: +{CHALLENGE.daily_target_pct}%/day, {CHALLENGE.daily_risk_pct}% wall | gamma/lambda LOCKED {GAMMA_LOCKED}/{LAMBDA_LOCKED}")
print(f"Reward weights: net_pnl={NET_PNL_WEIGHT} step_pnl={STEP_PNL_WEIGHT} daily_progress={DAILY_PROGRESS_WEIGHT} pain={DRAWDOWN_PAIN_WEIGHT} trade_quality={TRADE_QUALITY_WEIGHT}")
print(f"Aggression ranges: entropy={ENTROPY_RANGE} clip={CLIP_RANGE} lr={LR_RANGE} epochs={EPOCHS_RANGE} | rollout={ROLLOUT_SIZE}/{MINIBATCH_SIZE}")
print(f"OVERRIDES (diff vs baseline): {OVERRIDES if OVERRIDES else '{}  (pure baseline)'}")
print(f"Auto policy name: {POLICY_NAME}")

## 5. Train

This runs the **real PPO training loop** (milestone **M8** — `quantra/learning_system/trainer/trainer.py`):
each update **collects an on-policy rollout** → **GAE** (🔴 γ=0.997 / λ=0.97) → **K epochs of minibatch PPO**
(`ppo_loss`) → advances the **aggression scheduler** from the G8 missed-opportunity rate → **checkpoints** the brain.

Two cells below:
1. **Build the env** — loads your real MT5 bars via the data loader (Parquet cache → Drive → gdown-by-ID). If no
   bars are reachable (e.g. Drive not mounted), it falls back to a **clearly-labelled deterministic synthetic**
   series so the loop still runs as a smoke test. The env is built from the `CHALLENGE` / `REWARD_CFG` /
   `EPISODE_DAYS` objects from the HYPERPARAMETERS cell — nothing downstream is hardcoded.
2. **Train + checkpoint** — runs `Trainer.train(N_UPDATES)`, prints the per-update **training-health trace**
   (`approx_kl`, `clip_frac`, `value_loss`, `entropy`, `miss_rate`), proves the weights actually moved, and saves
   `artifacts/checkpoints/<policy>.pt` (tagged with `STATE_DIM` for the resume / compatibility gate) plus a
   `<policy>.history.json` trace.

> Raise `N_UPDATES` for a real run (50 is a quick proof). Watch `approx_kl` (should stay small / bounded) and
> `value_loss` (should stay finite) — those are the Risk Doctor's training-health signals. To **resume** a saved
> brain, load its `state_dict` into a fresh `PPOAgent` after the compatibility signature matches (see the
> Barbershop notebook Cell 4 for the gate).

In [ ]:
# ── 5a. Build the env the Trainer learns on (REAL bars; deterministic synthetic fallback) ──
# COUPLING -> quantra/market_pipeline/data_loader (load_symbol) + env/trading_env.py
#   (prepare_symbol_data, TradingEnv, SymbolData). The env consumes CHALLENGE / REWARD_CFG /
#   EPISODE_DAYS built in the HYPERPARAMETERS cell, so a knob change there changes what "passing"
#   means here — nothing is hardcoded downstream.
import numpy as np
from quantra.env.trading_env import TradingEnv, prepare_symbol_data, SymbolData

SYMBOL = "EURUSD"   # the binding case for real-bar training (see PROJECT_GUIDE §7)
try:
    from quantra.market_pipeline.data_loader import load_symbol
    df, rep = load_symbol(SYMBOL, path=None)                 # Parquet cache -> Drive -> gdown by ID
    SD = prepare_symbol_data(df, symbol=SYMBOL)              # builds the locked 207-feature obs (cached)
    usable = len(SD.close) - SD.valid_from
    if usable < 2000:
        raise RuntimeError(f"only {usable} warmed-up bars — too few to train")
    print(f"[data] REAL bars: {len(df):,}  warmup={SD.valid_from:,}  usable={usable:,}  source={rep.source}")
except Exception as e:
    # Deterministic synthetic fallback so the loop ALWAYS runs (smoke / verify). LOUDLY labelled, and
    # capped to a few days regardless of EPISODE_DAYS (synthetic is only for proving the loop turns).
    print(f"[data] real bars unavailable -> using DETERMINISTIC SYNTHETIC bars (smoke run only).\n        reason: {e}")
    from quantra.market_pipeline.feature_builder.schema import PRECOMPUTED_DIM, PRECOMPUTED_NAMES
    DAY = 1440; SYN_DAYS = min(max(EPISODE_DAYS, 4), 5); T = SYN_DAYS * DAY
    rng = np.random.default_rng(7)
    mat = np.zeros((T, PRECOMPUTED_DIM), dtype=np.float32)
    for nm, v in [("atr_dev_1m", .1), ("atr_dev_30m", .1), ("spread_range_ratio_1m", .3), ("adf_stat_1m", -3.5)]:
        mat[:, PRECOMPUTED_NAMES.index(nm)] = v             # open the 3 market-condition signals
    for nm in ("cci10_1m", "cci30_1m"):                      # a little signal for the policy to chew on
        if nm in PRECOMPUTED_NAMES:
            mat[:, PRECOMPUTED_NAMES.index(nm)] = rng.normal(0, 80, T)
    close = (1.20 * np.exp(np.cumsum(rng.normal(0, 4e-4, T)))).astype(float)
    SD = SymbolData(mat, close=close, atr=np.full(T, 1e-3), spread=np.full(T, 2e-5),
                    valid_from=0, dates=np.repeat(np.arange(T // DAY + 1), DAY)[:T])  # dates -> daily reset

# One env for the whole run; the account carries forward across EPISODE_DAYS (C10).
env = TradingEnv({SYMBOL: SD}, challenge=CHALLENGE, reward_cfg=REWARD_CFG, episode_days=EPISODE_DAYS)
print(f"[env] T={env.T:,} bars  episode_days={env.episode_days}  obs_width={len(env.reset())} (== STATE_DIM 207)")

In [ ]:
# ── 5b. THE PPO TRAINING LOOP — collect rollout -> GAE -> K-epoch minibatch PPO -> schedule -> checkpoint ──
# COUPLING -> quantra/learning_system/trainer/trainer.py (Trainer.train / .checkpoint),
#   ppo_agent/agent.py (PPOAgent reads STATE_DIM dynamically), trainer/scheduler.py
#   (AggressionScheduler picks dials within AGG_RANGES from the G8 miss-rate). The per-update line is the
#   TRAINING-HEALTH trace the Risk Doctor reads: approx_kl (step size), clip_frac (how often PPO clipped),
#   value_loss, entropy, miss_rate (G8 missed-opportunity rate).
import json, torch
from pathlib import Path
from quantra.learning_system.trainer.trainer import Trainer
from quantra.learning_system.trainer.scheduler import AggressionScheduler
from quantra.learning_system.ppo_agent.agent import PPOAgent

N_UPDATES = 50          # collect->update cycles. 50 = a quick proof; raise it (1000s) for a real run.

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
agent   = PPOAgent(device=DEVICE)                            # input dim read from schema (no hardcode)
trainer = Trainer(env, agent=agent, train_cfg=TRAIN_CFG,
                  scheduler=AggressionScheduler(AGG_RANGES), device=DEVICE)

print(f"Training on {DEVICE} · {N_UPDATES} updates · rollout={TRAIN_CFG.rollout_size}/{TRAIN_CFG.minibatch}\n")
_p0  = [p.detach().clone() for p in trainer.agent.net.parameters()]
hist = trainer.train(N_UPDATES)                              # the real loop
_every = max(1, N_UPDATES // 12)
for i, h in enumerate(hist):
    if i % _every == 0 or i == len(hist) - 1:
        print(f"  upd {i:4d}: kl={h['approx_kl']:+.4f} clip={h['clip_frac']:.3f} "
              f"value_loss={h['value_loss']:.4f} ent={h.get('entropy', 0):.3f} "
              f"miss={h['miss_rate']:.3f} lr={h['lr']:.1e} agg={h['aggression']:.2f}")

# Proof a real gradient step happened, and the health trace stayed sane.
_delta = sum((a - b).abs().sum().item() for a, b in zip(trainer.agent.net.parameters(), _p0))
_finite = all(np.isfinite(h['approx_kl']) and np.isfinite(h['value_loss']) for h in hist)
print(f"\n[learn] weights moved Σ|Δ|={_delta:.1f}  (a real gradient step happened: {_delta > 0})")
print(f"[health] all approx_kl & value_loss finite across the run: {_finite}")

# Checkpoint the brain (tagged with STATE_DIM for the resume / compatibility gate) + the health trace.
CKPT = trainer.checkpoint(POLICY_NAME)                      # -> artifacts/checkpoints/<POLICY_NAME>.pt
_hp = Path(CKPT).with_suffix(".history.json"); _hp.write_text(json.dumps(trainer.history, indent=2))
print(f"[ckpt] saved {CKPT}  (state_dim={STATE_DIM if 'STATE_DIM' in dir() else len(env.reset())})")
print(f"[hist] training-health trace -> {_hp}")
print("\nDone. Resume by loading this checkpoint's state_dict into a fresh PPOAgent once its compatibility")
print("signature matches (Barbershop Cell 4). Evaluate it with the Barbershop notebook (real per-day scoreboard).")